In [2]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"Au fost detectate {len(gpus)} GPU-uri active:")
    for i, gpu in enumerate(gpus):
        print(f" Placa {i}: {gpu.name}")
else:
    print("Niciun GPU detectat. Sistemul rulează lent pe CPU.")

Au fost detectate 2 GPU-uri active:
 Placa 0: /physical_device:GPU:0
 Placa 1: /physical_device:GPU:1


In [13]:
!pip install ai_edge_litert
from ai_edge_litert.interpreter import Interpreter

In [4]:
strategy = tf.distribute.MirroredStrategy()
print(f"Numărul de unități grafice sincronizate: {strategy.num_replicas_in_sync}")

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Numărul de unități grafice sincronizate: 2


In [5]:
from tensorflow import keras

modele = [
    "/kaggle/input/datasets/mindrocrobert/modele-emotii-faciale/mobilenet_gpu_fer-2013-augmented.keras",
    "/kaggle/input/datasets/mindrocrobert/modele-emotii-faciale/vcnn_gpu_ck.keras",
    "/kaggle/input/datasets/mindrocrobert/modele-emotii-faciale/vcnn_gpu_fer-2013-original.keras",
    "/kaggle/input/datasets/mindrocrobert/modele-emotii-faciale/vrescnn_gpu_raf-db.keras"
]

for path in modele:
    model = keras.models.load_model(path)
    print(f"{path.split('/')[-1]} → input shape: {model.input_shape}")

mobilenet_gpu_fer-2013-augmented.keras → input shape: (None, 48, 48, 1)
vcnn_gpu_ck.keras → input shape: (None, 48, 48, 1)
vcnn_gpu_fer-2013-original.keras → input shape: (None, 48, 48, 1)
vrescnn_gpu_raf-db.keras → input shape: (None, 48, 48, 1)


In [7]:
from tensorflow import keras
from tensorflow.keras.models import load_model
import tensorflow as tf

model = load_model('/kaggle/input/datasets/mindrocrobert/modele-emotii-faciale/mobilenet_gpu_fer-2013-augmented.keras')   
model.export('some', format="tf_saved_model")

mod=tf.saved_model.load('some')

INFO:tensorflow:Assets written to: some/assets


INFO:tensorflow:Assets written to: some/assets


Saved artifact at 'some'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 48, 48, 1), dtype=tf.float32, name='input_layer_4')]
Output Type:
  TensorSpec(shape=(None, 7), dtype=tf.float32, name=None)
Captures:
  135479049484624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135477918990416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135477918999824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135477918990608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135477919000400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135477918990992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135477918990800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135477919000208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135477918991376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135477919001360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135477919000784: Tens

In [8]:
converter = tf.lite.TFLiteConverter.from_keras_model(mod)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

tflite_file='mobilenet_gpu_fer-2013-augmented.tflite'
with open(tflite_file, 'wb') as f:
    f.write(tflite_model)

print("Conversia a fost realizată cu succes și modelul a fost salvat ca: ",tflite_file)

INFO:tensorflow:Assets written to: /tmp/tmp2fk88_8l/assets


INFO:tensorflow:Assets written to: /tmp/tmp2fk88_8l/assets


Conversia a fost realizată cu succes și modelul a fost salvat ca:  mobilenet_gpu_fer-2013-augmented.tflite


W0000 00:00:1778162183.242629      57 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1778162183.242691      57 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1778162183.299684      57 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled


In [11]:
interpreter = Interpreter(model_path="mobilenet_gpu_fer-2013-augmented.tflite") 
interpreter.allocate_tensors()

In [14]:
# celula 1 realizeaza preprocesarea datelor + augumentare
import keras
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_dir= '/kaggle/input/datasets/prilia/fer2013pluscleanedaugmballanced1/train'
test_dir='/kaggle/input/datasets/prilia/fer2013pluscleanedaugmballanced1/test'

print ("Versiune TensorFlow: ", tf.__version__)
print ("Versiune Keras: ", keras.__version__)

img_size=48
batch_size=64
epoci = 50
num_classes= 7 
print ("\n partea de preprocesare si impartire a datelor")
datagen = ImageDataGenerator (
    featurewise_center=False,
    featurewise_std_normalization=False,
    rescale=1./255,
    rotation_range=10,
    zoom_range= 0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8,1.2],
    fill_mode='nearest',
    validation_split=0.20
)

test_datagen = ImageDataGenerator(rescale=1./255)

print("Setul de antrenare (80%):")
train_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    color_mode="grayscale",      
    batch_size=batch_size,
    class_mode='categorical',     
    shuffle=True                  
)

print("Setul de validare (20%):")
val_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    color_mode="grayscale",
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False,                
    subset='validation'           
)

print("Setul de testare:")
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_size, img_size),
    color_mode="grayscale",
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)


Versiune TensorFlow:  2.19.0
Versiune Keras:  3.10.0

 partea de preprocesare si impartire a datelor
Setul de antrenare (80%):
Found 23251 images belonging to 7 classes.
Setul de validare (20%):
Found 4648 images belonging to 7 classes.
Setul de testare:
Found 5772 images belonging to 7 classes.


In [22]:
import numpy as np
import time
from sklearn.metrics import classification_report, confusion_matrix

input_index = interpreter.get_input_details()[0]["index"]
output_index = interpreter.get_output_details()[0]["index"]

test_generator.reset()
y_true = test_generator.classes
predictions = []
durata_totala = 0

for img_batch, _ in test_generator:
    for img in img_batch:
        img_exp = np.expand_dims(img, axis=0).astype(np.float32)
        interpreter.set_tensor(input_index, img_exp)
        t1 = time.time()
        interpreter.invoke()
        t2 = time.time()
        durata_totala += (t2 - t1)
        output = interpreter.get_tensor(output_index)
        predictions.append(np.argmax(output))
    if len(predictions) >= len(y_true):
        break

predictions = predictions[:len(y_true)]
acc = sum(p == t for p, t in zip(predictions, y_true)) / len(y_true)
latenta = 1000 * durata_totala / len(y_true)

print(f" Acuratețe MobileNet FER-2013 (.tflite): {acc*100:.2f}%")
print(f" Latența per imagine: {latenta:.4f} ms")
cm = confusion_matrix(y_true, predictions)
print(cm)
etichete = list(test_generator.class_indices.keys())
print(classification_report(y_true, predictions, target_names=etichete))

 Acuratețe MobileNet FER-2013 (.tflite): 80.91%
 Latența per imagine: 1.2182 ms
[[683   0  15  22  46  26  36]
 [ 27 794   3   5   4   6   2]
 [ 12   4 749   2   6  17  46]
 [ 40   1   6 679  49  26  32]
 [ 45   0   8  32 546 119  56]
 [ 57   4  15  34 137 518  33]
 [ 26   0  25  27  37  14 701]]
              precision    recall  f1-score   support

       Anger       0.77      0.82      0.80       828
     Disgust       0.99      0.94      0.97       841
        Fear       0.91      0.90      0.90       836
       Happy       0.85      0.82      0.83       833
     Neutral       0.66      0.68      0.67       806
         Sad       0.71      0.65      0.68       798
    Surprise       0.77      0.84      0.81       830

    accuracy                           0.81      5772
   macro avg       0.81      0.81      0.81      5772
weighted avg       0.81      0.81      0.81      5772



In [27]:
import os

size_in_bytes = os.path.getsize('mobilenet_gpu_fer-2013-augmented.tflite')
lite_sz = round(size_in_bytes/1000, 2)

print('Raport final MobileNet FER-2013:', 
      'Lite_sz', str(lite_sz)+'k',
      'acc:', str(round(100*acc, 2)),
      'lat', str(round(latenta, 2)))

Raport final MobileNet FER-2013: Lite_sz 3652.91k acc: 80.91 lat 1.22
